In [3]:
#(1) captura do frame e aplicação do undistort
import cv2
import numpy as np
from dnn_utils import read_pipeline_frame, DEFAULT_K, DEFAULT_DIST, OUT

frame, path = read_pipeline_frame()
undist = cv2.undistort(frame, DEFAULT_K, DEFAULT_DIST)
painel = np.hstack([frame, undist])
cv2.putText(painel, 'original', (20,40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,0), 2)
cv2.putText(painel, 'undistort', (frame.shape[1]+20,40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,0), 2)
cv2.imwrite(str(OUT/'undistort_painel.jpg'), painel)
print('Painel salvo em saidas/undistort_painel.jpg')

Painel salvo em saidas/undistort_painel.jpg


In [4]:
#(2) segmente ROI por cor HSV
import cv2
from dnn_utils import read_pipeline_frame, segment_red_hsv, OUT

frame, _ = read_pipeline_frame()
mask, roi = segment_red_hsv(frame)
vis = frame.copy()

if roi:
    x,y,w,h = roi
    cv2.rectangle(vis, (x,y), (x+w,y+h), (0,255,255), 3)
    cv2.putText(vis, 'ROI HSV', (x,y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,0), 2)
else:
    print('Nenhuma ROI encontrada.')

cv2.imwrite(str(OUT/'roi_hsv.jpg'), vis)
print('Resultados salvos em saidas/. ROI:', roi)

Resultados salvos em saidas/. ROI: (61, 224, 213, 195)


In [5]:
#(3) extraia features ORB
import cv2
from dnn_utils import read_pipeline_frame, segment_red_hsv, orb_features, OUT

frame, _ = read_pipeline_frame()
mask, roi = segment_red_hsv(frame)

if roi:
    x,y,w,h = roi
    crop = frame[y:y+h, x:x+w]
else:
    crop = frame

kp, des = orb_features(crop)
vis = cv2.drawKeypoints(crop, kp, None, color=(0,255,0), flags=0)
cv2.imwrite(str(OUT/'orb_roi.jpg'), vis)
print('Quantidade de keypoints ORB:', len(kp))
print('Descritores:', None if des is None else des.shape)

Quantidade de keypoints ORB: 84
Descritores: (84, 32)


In [6]:
#(4) aplique detector HOG+SVM ou Haar Cascade
import cv2
from dnn_utils import read_pipeline_frame, hog_or_haar_detector, OUT

frame, _ = read_pipeline_frame()
boxes = hog_or_haar_detector(frame)
vis = frame.copy()

if boxes:
    for nome,x,y,w,h in boxes:
        cv2.rectangle(vis, (x,y), (x+w,y+h), (255,0,0), 2)
        cv2.putText(vis, nome, (x,y-8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,0,0), 2)
else:
    cv2.putText(vis, 'HOG/Haar executado: sem deteccao neste frame', (30,40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,180), 2)

cv2.imwrite(str(OUT/'detector_hog_haar.jpg'), vis)
print('Caixas detectadas:', boxes)

Caixas detectadas: [('Haar', 55, 120, 321, 321)]


In [ ]:
#(5) classifique a ROI detectada com o modelo DNN do item A. 
import cv2
from dnn_utils import read_pipeline_frame, segment_red_hsv, predict_opencv, topk, draw_top3, OUT

frame, _ = read_pipeline_frame()
mask, roi = segment_red_hsv(frame)

if roi:
    x,y,w,h = roi
    crop = frame[y:y+h, x:x+w]
else:
    crop = frame
    x,y,w,h = 0,0,frame.shape[1], frame.shape[0]

prob = predict_opencv(crop)
vis = draw_top3(frame, topk(prob, 3))
cv2.rectangle(vis, (x,y), (x+w,y+h), (0,255,255), 3)
cv2.imwrite(str(OUT/'classificar_roi.jpg'), vis)
print('ROI classificada e anotada.')
